# 🎭 Kana Arima DPO Fine-Tuning with Unsloth

This notebook trains a roleplay character model using **Direct Preference Optimization (DPO)** on 359 preference pairs from **My C.AI Universe** Tournament.

- **Hardware**: Google Colab Free T4 GPU (or L4/A100)
- **Base Model**: `unsloth/Llama-3.2-3B-Instruct` (or `unsloth/Qwen2.5-7B-Instruct`)
- **Local GPU Usage**: 0% (Everything runs on Google Cloud)
- **Estimated Duration**: ~6–10 minutes

### Step 1: Install Unsloth & Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

### Step 2: Upload Datasets
Upload `kana_dpo_train.jsonl` and `kana_dpo_eval.jsonl` from your `data/training/` folder.

In [ ]:
from google.colab import files
import os

if not os.path.exists('kana_dpo_train.jsonl'):
    print("Please upload kana_dpo_train.jsonl and optionally kana_dpo_eval.jsonl:")
    uploaded = files.upload()

### Step 3: Load Model with Unsloth 4-bit Quantization

In [ ]:
import torch
from unsloth import FastLanguageModel, PatchDPOTrainer
from datasets import Dataset
import json

max_seq_length = 2048
base_model = "unsloth/Llama-3.2-3B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# Attach LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

PatchDPOTrainer()
tokenizer.pad_token = tokenizer.eos_token

### Step 4: Prepare Datasets for DPOTrainer

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return Dataset.from_list(rows)

train_ds = load_jsonl('kana_dpo_train.jsonl')
eval_ds = load_jsonl('kana_dpo_eval.jsonl') if os.path.exists('kana_dpo_eval.jsonl') else None
print(f"Loaded {len(train_ds)} training pairs, {len(eval_ds) if eval_ds else 0} eval pairs.")

### Step 5: Configure and Run DPO Training

In [ ]:
from trl import DPOTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_ratio = 0.1,
    num_train_epochs = 3,
    learning_rate = 5e-6,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 5,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",
    output_dir = "kana_dpo_output",
    save_strategy = "epoch",
    report_to = "none"
)

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,
    train_dataset = train_ds,
    eval_dataset = eval_ds,
    tokenizer = tokenizer,
    max_length = max_seq_length,
    max_prompt_length = max_seq_length // 2,
    args = training_args,
)

dpo_trainer.train()

### Step 6: Test Character Voice Inference

In [ ]:
FastLanguageModel.for_inference(model)

sample_messages = train_ds[0]['prompt']
inputs = tokenizer.apply_chat_template(
    sample_messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 150, temperature = 0.7, use_cache = True)
print(tokenizer.batch_decode(outputs)[0])

### Step 7: Export to GGUF (for Ollama) and Save to Google Drive

In [ ]:
# Export GGUF Q4_K_M for Ollama
model.save_pretrained_gguf("kana_arima_dpo", tokenizer, quantization_method = "q4_k_m")

# Or save 16-bit merged model
model.save_pretrained_merged("kana_arima_dpo_merged", tokenizer, save_method = "merged_16bit")

print("Model saved successfully!")